# Amenity Feature Engineering

**Run once.** Outputs `Data/processed/amenity_features.parquet` — merge on `id` into any downstream notebook.

| Phase | Purpose |
|---|---|
| 1 | Parse & Inventory — frequency table for all amenities |
| 2 | Individual Signal — price lift + correlation per amenity |
| 3 | Bundle Discovery — co-occurrence clustering |
| 4 | Final Selection — score & pick flags |
| 5 | Engineer & Save — `amenity_count` + binary flags → parquet |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import ast, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import MultiLabelBinarizer

RANDOM_STATE = 42
DATA_PATH    = "../Data/processed/listings_all_cities.parquet"

TOP_N        = 60    # amenities to carry into binary matrix
MIN_FREQ     = 0.05  # drop amenities present in <5% of listings

print('Setup complete.')

## Phase 1 — Parse & Inventory

In [ ]:
COLS = ['id', 'city', 'price', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'amenities']
df = pd.read_parquet(DATA_PATH, columns=COLS)

def parse_amenities(s):
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

df['amenity_list']  = df['amenities'].apply(parse_amenities)
df['amenity_count'] = df['amenity_list'].apply(len)

print(f"Listings:              {len(df):,}")
print(f"Amenity count — mean:  {df['amenity_count'].mean():.1f}")
print(f"Amenity count — median:{df['amenity_count'].median():.0f}")
print(f"Amenity count — max:   {df['amenity_count'].max()}")
df['amenity_count'].describe().round(1)

In [ ]:
from collections import Counter

all_amenities = [a for lst in df['amenity_list'] for a in lst]
freq_counter  = Counter(all_amenities)

freq_df = (
    pd.DataFrame(freq_counter.items(), columns=['amenity', 'count'])
    .assign(frequency=lambda d: d['count'] / len(df))
    .sort_values('frequency', ascending=False)
    .reset_index(drop=True)
)

print(f"Total unique amenities:           {len(freq_df):,}")
print(f"Amenities ≥ {MIN_FREQ:.0%} frequency:       {(freq_df['frequency'] >= MIN_FREQ).sum()}")
print(f"Amenities ≥  1% frequency:        {(freq_df['frequency'] >= 0.01).sum()}")

freq_df.head(20)

In [ ]:
top30 = freq_df.head(30)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top30['amenity'][::-1], top30['frequency'][::-1] * 100, color='steelblue')
ax.set_xlabel('% of listings')
ax.set_title('Top 30 amenities by frequency')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Distribution of amenity count per listing
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(df['amenity_count'], bins=50, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(df['amenity_count'].median(), color='red', linestyle='--', label='median')
ax.set_xlabel('Number of amenities per listing')
ax.set_title('Distribution of amenity count')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Build binary presence matrix for top-N amenities
top_amenities = freq_df[freq_df['frequency'] >= MIN_FREQ].head(TOP_N)['amenity'].tolist()

mlb = MultiLabelBinarizer(classes=top_amenities)
bin_arr = mlb.fit_transform(df['amenity_list'])
bin_df  = pd.DataFrame(bin_arr, columns=top_amenities, index=df.index)

print(f"Binary matrix shape: {bin_df.shape}")
print(f"Amenities included:  {len(top_amenities)}")

## Phase 2 — Individual Amenity Signal

For each amenity compute:
- **Price lift** = mean price (with amenity) / mean price (without) — values > 1 mean the amenity correlates with higher prices
- **Point-biserial correlation** with `price` and `estimated_occupancy_l365d`
- **Combined score** = z-score(price_lift) × frequency percentile — penalises very rare amenities

In [ ]:
records = []
for amenity in top_amenities:
    mask    = bin_df[amenity] == 1
    n_with  = mask.sum()
    n_total = len(df)

    if n_with < 50:
        continue

    mean_price_with    = df.loc[mask,  'price'].mean()
    mean_price_without = df.loc[~mask, 'price'].mean()
    price_lift         = mean_price_with / mean_price_without if mean_price_without > 0 else np.nan

    mean_occ_with    = df.loc[mask,  'estimated_occupancy_l365d'].mean()
    mean_occ_without = df.loc[~mask, 'estimated_occupancy_l365d'].mean()
    occ_lift         = mean_occ_with / mean_occ_without if mean_occ_without > 0 else np.nan

    price_corr, price_pval = stats.pointbiserialr(bin_df[amenity], df['price'])
    occ_corr,   occ_pval   = stats.pointbiserialr(bin_df[amenity], df['estimated_occupancy_l365d'])

    records.append({
        'amenity':          amenity,
        'frequency':        n_with / n_total,
        'n_with':           n_with,
        'mean_price_with':  round(mean_price_with, 1),
        'mean_price_wo':    round(mean_price_without, 1),
        'price_lift':       round(price_lift, 3),
        'price_corr':       round(price_corr, 3),
        'price_pval':       price_pval,
        'occ_lift':         round(occ_lift, 3),
        'occ_corr':         round(occ_corr, 3),
        'occ_pval':         occ_pval,
    })

signal_df = pd.DataFrame(records)

# Combined score: absolute price correlation × frequency percentile
signal_df['freq_pct']      = signal_df['frequency'].rank(pct=True)
signal_df['abs_price_corr']= signal_df['price_corr'].abs()
signal_df['combined_score']= signal_df['abs_price_corr'] * signal_df['freq_pct']
signal_df = signal_df.sort_values('combined_score', ascending=False).reset_index(drop=True)

print(f"Amenities analysed: {len(signal_df)}")
signal_df[['amenity','frequency','price_lift','price_corr','occ_lift','occ_corr','combined_score']].head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Price lift vs frequency ──────────────────────────────────────────────────
ax = axes[0]
sc = ax.scatter(
    signal_df['frequency'] * 100,
    signal_df['price_lift'],
    c=signal_df['combined_score'],
    cmap='RdYlGn', s=60, alpha=0.8, edgecolors='grey', linewidths=0.3
)
plt.colorbar(sc, ax=ax, label='combined score')
ax.axhline(1, color='grey', linestyle='--', alpha=0.5, label='no lift')

# Label top-20 by combined score
for _, row in signal_df.head(20).iterrows():
    ax.annotate(row['amenity'], (row['frequency'] * 100, row['price_lift']),
                fontsize=6, ha='left', va='bottom',
                xytext=(3, 2), textcoords='offset points')

ax.set_xlabel('Frequency (% of listings)')
ax.set_ylabel('Price lift (mean price with / without)')
ax.set_title('Price lift vs frequency')
ax.grid(True, alpha=0.3)

# ── Point-biserial: price vs occupancy correlation ───────────────────────────
ax = axes[1]
sc2 = ax.scatter(
    signal_df['price_corr'],
    signal_df['occ_corr'],
    c=signal_df['combined_score'],
    cmap='RdYlGn', s=60, alpha=0.8, edgecolors='grey', linewidths=0.3
)
plt.colorbar(sc2, ax=ax, label='combined score')
ax.axhline(0, color='grey', linestyle='--', alpha=0.5)
ax.axvline(0, color='grey', linestyle='--', alpha=0.5)

for _, row in signal_df.head(20).iterrows():
    ax.annotate(row['amenity'], (row['price_corr'], row['occ_corr']),
                fontsize=6, ha='left', va='bottom',
                xytext=(3, 2), textcoords='offset points')

ax.set_xlabel('Point-biserial corr with price')
ax.set_ylabel('Point-biserial corr with occupancy')
ax.set_title('Price signal vs occupancy signal')
ax.grid(True, alpha=0.3)

plt.suptitle('Amenity signal analysis', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: price lift for top-25 by combined score
top25 = signal_df.head(25).sort_values('price_lift')

colors = ['#d6604d' if v > 1 else '#4393c3' for v in top25['price_lift']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top25['amenity'], top25['price_lift'], color=colors, alpha=0.85)
ax.axvline(1, color='black', linestyle='--', linewidth=1, label='baseline (lift = 1)')

for bar, val in zip(bars, top25['price_lift']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}×', va='center', fontsize=7)

ax.set_xlabel('Price lift (mean price with / mean price without)')
ax.set_title('Top 25 amenities by combined score — price lift')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Price lift by city for the top amenities — does signal generalise across cities?
top_flag_candidates = signal_df.head(15)['amenity'].tolist()

city_lift = {}
for city in df['city'].unique():
    mask_city = df['city'] == city
    lifts = {}
    for amenity in top_flag_candidates:
        mask_a  = bin_df.loc[mask_city.values, amenity] == 1
        with_ = df.loc[mask_city & (bin_df[amenity] == 1).values, 'price'].mean()
        wo_   = df.loc[mask_city & (bin_df[amenity] == 0).values, 'price'].mean()
        lifts[amenity] = round(with_ / wo_, 3) if wo_ > 0 else np.nan
    city_lift[city] = lifts

city_lift_df = pd.DataFrame(city_lift).T[top_flag_candidates]
city_lift_df.index.name = 'city'

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    city_lift_df,
    annot=True, fmt='.2f', cmap='RdYlGn', center=1.0,
    linewidths=0.5, ax=ax, cbar_kws={'label': 'price lift'}
)
ax.set_title('Price lift per amenity × city (values > 1 = premium signal)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## Phase 3 — Bundle Discovery

Amenities that co-occur form natural **bundles** (e.g. a well-equipped kitchen bundle, a luxury comfort bundle).  
Method:
1. Compute pairwise **Jaccard similarity** between the top-N amenity columns.
2. Hierarchical clustering (Ward linkage on `1 − Jaccard`) to reveal groups.
3. Cut dendrogram at a depth that yields 5–8 interpretable bundles.
4. Validate each bundle by mean price of listings that have **all** bundle members.

In [ ]:
# Jaccard similarity matrix between amenity columns
# Jaccard(A,B) = |A ∩ B| / |A ∪ B|
X = bin_df.values.astype(np.float32)   # (N_listings × N_amenities)

intersection = X.T @ X                              # (N_amenities × N_amenities)
col_sums     = X.sum(axis=0, keepdims=True)         # (1 × N_amenities)
union        = col_sums + col_sums.T - intersection

jaccard_sim  = np.where(union > 0, intersection / union, 0)
np.fill_diagonal(jaccard_sim, 1.0)

jaccard_sim_df = pd.DataFrame(jaccard_sim, index=top_amenities, columns=top_amenities)
dist_matrix    = 1 - jaccard_sim

print("Jaccard similarity matrix shape:", jaccard_sim_df.shape)
print("\nTop 5 most similar pairs:")
sim_pairs = []
for i in range(len(top_amenities)):
    for j in range(i+1, len(top_amenities)):
        sim_pairs.append((top_amenities[i], top_amenities[j], jaccard_sim[i,j]))
sim_pairs.sort(key=lambda x: -x[2])
for a, b, s in sim_pairs[:10]:
    print(f"  {s:.3f}  {a}  ↔  {b}")

In [ ]:
# Ward linkage on distance matrix
condensed_dist = squareform(dist_matrix, checks=False)
Z = linkage(condensed_dist, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(
    Z,
    labels=top_amenities,
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=0.6,
    ax=ax,
)
ax.set_title('Amenity dendrogram — Ward linkage on Jaccard distance', fontsize=12)
ax.set_ylabel('Distance')
ax.axhline(0.6, color='red', linestyle='--', alpha=0.6, label='cut at 0.6')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Reorder heatmap by cluster order from dendrogram
N_BUNDLES    = 7  # adjust after inspecting dendrogram above
bundle_labels = fcluster(Z, N_BUNDLES, criterion='maxclust')
order         = np.argsort(bundle_labels)
ordered_ams   = [top_amenities[i] for i in order]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    jaccard_sim_df.loc[ordered_ams, ordered_ams],
    cmap='YlOrRd', vmin=0, vmax=0.6,
    linewidths=0.3, linecolor='white',
    xticklabels=True, yticklabels=True,
    ax=ax, cbar_kws={'label': 'Jaccard similarity'}
)
ax.set_title('Amenity co-occurrence (Jaccard) — clustered order', fontsize=12)
ax.tick_params(axis='x', labelsize=7, rotation=45)
ax.tick_params(axis='y', labelsize=7, rotation=0)
plt.tight_layout()
plt.show()

# Print bundle membership
bundle_map = pd.Series(bundle_labels, index=top_amenities)
for b in sorted(bundle_map.unique()):
    members = bundle_map[bundle_map == b].index.tolist()
    print(f"Bundle {b}: {members}")

In [ ]:
# Validate bundles: mean price / occupancy for listings that have ≥50% of each bundle
bundle_stats = []
for b in sorted(bundle_map.unique()):
    members = bundle_map[bundle_map == b].index.tolist()
    # listing has bundle if it has ≥50% of member amenities
    bundle_presence = (bin_df[members].sum(axis=1) / len(members)) >= 0.5
    n_with  = bundle_presence.sum()
    mean_p  = df.loc[bundle_presence.values, 'price'].mean()
    mean_o  = df.loc[bundle_presence.values, 'estimated_occupancy_l365d'].mean()
    global_p = df['price'].mean()
    global_o = df['estimated_occupancy_l365d'].mean()
    bundle_stats.append({
        'bundle':        b,
        'n_members':     len(members),
        'n_listings':    int(n_with),
        'pct_listings':  round(n_with / len(df) * 100, 1),
        'mean_price':    round(mean_p, 1),
        'price_lift':    round(mean_p / global_p, 3),
        'mean_occ':      round(mean_o, 3),
        'occ_lift':      round(mean_o / global_o, 3),
        'members':       ', '.join(members[:5]) + ('...' if len(members) > 5 else ''),
    })

bundle_stats_df = pd.DataFrame(bundle_stats).sort_values('price_lift', ascending=False)
print(f"Global mean price:      {df['price'].mean():.1f}")
print(f"Global mean occupancy:  {df['estimated_occupancy_l365d'].mean():.3f}\n")
bundle_stats_df

## Phase 4 — Final Feature Selection

Based on the analysis above, select:
1. **`amenity_count`** — continuous, already computed
2. **Individual flags** — highest combined-score amenities that also generalise across cities
3. **Bundle flag(s)** — any bundle whose mean price lift > 1.15 AND covers > 10% of listings

Naming convention: `has_<snake_case>` for individual flags, `bundle_<name>` for bundles.

> **Update `SELECTED_FLAGS` below** after reviewing Phases 2 & 3.

In [ ]:
# ── TUNE THESE ────────────────────────────────────────────────────────────────
TOP_N_PRICE = 5    # top N amenities by |price correlation|
TOP_N_OCC   = 5    # top N amenities by |occupancy correlation|
# Uses absolute correlation so strongly negative signals (e.g. "Pets allowed",
# "Long term stays") are selected alongside strongly positive ones.

MIN_BUNDLE_PRICE_LIFT = 1.10   # include a bundle if its price_lift >= this
MIN_BUNDLE_PCT_LISTINGS = 10.0 # AND it covers >= this % of listings

# ── Individual flag auto-selection ────────────────────────────────────────────
signal_df['abs_price_corr'] = signal_df['price_corr'].abs()
signal_df['abs_occ_corr']   = signal_df['occ_corr'].abs()

top_by_price = signal_df.nlargest(TOP_N_PRICE, 'abs_price_corr')['amenity'].tolist()
top_by_occ   = signal_df.nlargest(TOP_N_OCC,   'abs_occ_corr')['amenity'].tolist()

# Union, price order first, then occ-only additions
selected = list(dict.fromkeys(top_by_price + top_by_occ))

def to_flag_name(amenity):
    name = re.sub(r'[^a-z0-9]+', '_', amenity.lower()).strip('_')
    return f'has_{name}'

INDIVIDUAL_FLAGS = {to_flag_name(a): a for a in selected}

# ── Bundle auto-selection ─────────────────────────────────────────────────────
qualifying = bundle_stats_df[
    (bundle_stats_df['price_lift']   >= MIN_BUNDLE_PRICE_LIFT) &
    (bundle_stats_df['pct_listings'] >= MIN_BUNDLE_PCT_LISTINGS)
]['bundle'].tolist()

# ── RENAME MAP: fill in after reviewing Phase 3 dendrogram / bundle table ─────
# Keys are auto-names ('bundle_1', 'bundle_3', …), values are your chosen names.
BUNDLE_RENAME = {
    # 'bundle_2': 'bundle_full_kitchen',
    # 'bundle_5': 'bundle_premium_comfort',
}

BUNDLE_FLAGS = {}
for b in qualifying:
    members    = bundle_map[bundle_map == b].index.tolist()
    auto_name  = f'bundle_{b}'
    final_name = BUNDLE_RENAME.get(auto_name, auto_name)
    BUNDLE_FLAGS[final_name] = members

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"── Individual flags ({len(INDIVIDUAL_FLAGS)}) ─────────────────────────────")
for flag, amenity in INDIVIDUAL_FLAGS.items():
    row = signal_df[signal_df['amenity'] == amenity].iloc[0]
    sources = []
    if amenity in top_by_price:
        direction = '▲' if row['price_corr'] > 0 else '▼'
        sources.append(f'price_corr={row["price_corr"]:+.3f} {direction}')
    if amenity in top_by_occ:
        direction = '▲' if row['occ_corr'] > 0 else '▼'
        sources.append(f'occ_corr={row["occ_corr"]:+.3f} {direction}')
    print(f"  {flag:<42}  {', '.join(sources)}")

print(f"\n── Bundle flags ({len(BUNDLE_FLAGS)}) ───────────────────────────────────")
for name, members in BUNDLE_FLAGS.items():
    row = bundle_stats_df[bundle_stats_df['bundle'] == int(name.split('_')[-1])].iloc[0]
    print(f"  {name}  (price_lift={row['price_lift']:.3f}, {row['pct_listings']:.1f}% of listings)")
    print(f"    {members}")

In [ ]:
# Full signal table for selected individual flags
rows = signal_df[signal_df['amenity'].isin(INDIVIDUAL_FLAGS.values())].copy()
rows = rows.merge(
    pd.DataFrame({'amenity': list(INDIVIDUAL_FLAGS.values()),
                  'flag_name': list(INDIVIDUAL_FLAGS.keys())}),
    on='amenity'
).sort_values('price_lift', ascending=False)

display_cols = ['flag_name', 'amenity', 'frequency', 'price_lift',
                'price_corr', 'occ_lift', 'occ_corr', 'combined_score']
rows[display_cols].round(3)

## Phase 5 — Engineer & Save

Build the final feature table and write to `Data/processed/amenity_features.parquet`.  
Downstream notebooks merge on `id`:
```python
amenity_feats = pd.read_parquet('../Data/processed/amenity_features.parquet')
df = df.merge(amenity_feats, on='id', how='left')
```

In [ ]:
feat_df = pd.DataFrame({'id': df['id'].values, 'amenity_count': df['amenity_count'].values})

# Individual flags — exact membership check (reliable, no partial-match issues)
amenity_sets = df['amenity_list'].apply(set)

for flag_name, raw_amenity in INDIVIDUAL_FLAGS.items():
    feat_df[flag_name] = amenity_sets.apply(lambda s: int(raw_amenity in s)).values

# Bundle flags — ≥50% of bundle members present
for bundle_name, members in BUNDLE_FLAGS.items():
    feat_df[bundle_name] = amenity_sets.apply(
        lambda s: int(sum(m in s for m in members) / len(members) >= 0.5)
    ).values

print(f"Feature table shape: {feat_df.shape}")
print(f"\nColumn means:")
print(feat_df.drop(columns='id').mean().round(3).to_string())

In [ ]:
# Quick cross-check: amenity_count distribution by city
city_counts = df.groupby('city')['amenity_count'].describe().round(1)
print("amenity_count by city:")
print(city_counts)

# Correlation between amenity_count and price
corr_cnt, _ = stats.pearsonr(feat_df['amenity_count'], df['price'])
print(f"\nPearson corr(amenity_count, price): {corr_cnt:.3f}")

In [ ]:
# Read the full parquet, drop any stale amenity columns, patch in the new ones, write back
df_full = pd.read_parquet(DATA_PATH)

stale = [c for c in df_full.columns if c == 'amenity_count' or c.startswith('has_') or c.startswith('bundle_')]
if stale:
    print(f"Dropping stale columns: {stale}")
    df_full = df_full.drop(columns=stale)

new_cols = [c for c in feat_df.columns if c != 'id']
df_full  = df_full.merge(feat_df, on='id', how='left')

df_full.to_parquet(DATA_PATH, index=False)

print(f"Updated: {DATA_PATH}")
print(f"New shape: {df_full.shape}  (added {len(new_cols)} columns)")
print(f"\nNew columns added:")
for col in new_cols:
    null = df_full[col].isna().sum()
    print(f"  {col:<35} nulls={null}")

## How to use in downstream notebooks

The amenity columns are now part of `listings_all_cities.parquet` — no merge needed.

**In `segmentation.ipynb` Phase 6**, add to the feature lists:
```python
# amenity_count → CONTINUOUS_FEATURES (already log1p candidate)
# has_* flags   → BINARY_FEATURES
AMENITY_CONTINUOUS = ['amenity_count']
AMENITY_FLAGS      = ['has_elevator', 'has_ac', 'has_dishwasher', 'has_washer',
                      'has_self_checkin', 'has_balcony', 'has_private_balcony',
                      'has_bathtub', 'has_dedicated_workspace',
                      'has_long_term_stays', 'has_pets_allowed', 'has_parking']
# append to CONTINUOUS_FEATURES and BINARY_FEATURES before building X
```

**Re-run this notebook** any time you want to change which flags are included — it drops stale `amenity_count` / `has_*` / `bundle_*` columns before writing, so it is safe to re-run.